## Details covered below:
## 1. The mystery of <<1 ZZ coherences for ATaCR on some stations (Z6.11).
###     Spoiler: It was a mix of bad noise data and an fft-window used in my coherences that was too short, producing spectral leakage.
## 2. A comprehensive source of details on the reasons for every day, station, and source-receiver pair that has been killed thus far.



## Stem plots:
### - [LINK : Notched Stem Plots](znb_images/UpdatedData_1.31.25/_notched_stem_plots.pdf)
### - [LINK : Banded Stem Plots](znb_images/UpdatedData_1.31.25/_banded_stem_plots.pdf)

## 1. The mystery of <<1 ZZ coherences for ATaCR on some stations (Z6.11 and 7D.FS08D).

## The problem: Z6.11, and to a lesser extent 7D.G17B, exhibit ZZ coherences siginificantly less than 1.0 post-ATaCR at frequencies higher than their respective notch.
## Station ZZ coherences for each demonstrating this:
<img src="./NB_Assets/04.05.25/_depreciated.Z6.11.both.Coherence.png" alt="A" width="600">
<img src="./NB_Assets/04.05.25/_depreciated.7D.G17B.both.Coherence.png" alt="A" width="600">

### Set of things to investigate:
## 1. Transfer functions. Does it taper to zero post-notch?
## 2. Does the tapering of the transfer function produce Gibb's phenomena as a result of too-steeply (aka brick-walling) tapering?
## 3. Bad noise?
## 4. Bad event observations?

### [1 and 2] Two the first two items (TF and TF tapers):

I have confirmed the taper is zeroed out perfectly by the notch for every single station. 

To the extent of steepness, I have tried:
-Reducing the taper slope to small (taper widths from 50 to 100 samples) and extreme amounts (taper widths from 50 to 300 samples). For every single one, nothing changed. 

I have also tried different species of taper design: 
-from tapered Cosine (the default in ATaCR), 
-linear slope,root-function, 
-and spheroidal-prolate (Slepian with peak flattened to 1.0) tapers. 

For every single one, nothing changed. Showing plots for each is a bit redundant as they all looked EXACTLY the same as above. So, there is no indication at all that this is a resulf of the TF or TF-tapers.


## Transfer functions with the default taper:

<img src="./NB_Assets/04.05.25/04.05.Z6.11.transfer_functions.png" alt="A" width="600">
<img src="./NB_Assets/04.05.25/04.05.7D.G17B.transfer_functions.png" alt="A" width="600">

[3.] Bad Noise? Most likely, if not completely, the cause of the issue.

## Below are plots of the current set of noise collected for these two stations organized (from left to right) in columns of good, bad, and redundant days of noise. Several of the days listed under 'Quarantined' were originally listed by ATaCR as good days when this issue above was found. Furthermore, all of the days listed under 'extra days' (3rd column) were still being used by ATaCR. This has since been corrected and ATaCR uses 10, and only 10, days of noise for each station. 

<img src="./NB_Assets/04.05.25/Z6.11.Noise.QC.png" alt="A" width="600">

<img src="./NB_Assets/04.05.25/7D.G17B.Noise.QC.png" alt="A" width="600">

## I think most of the noise shown for Z6.11 under quarantine is perfectly fine, but it wasn't until I replaced them with some of the extra days I had that my ZZ coherences went 

## from this (repeated plots from above):
<img src="./NB_Assets/04.05.25/_depreciated.Z6.11.both.Coherence.png" alt="A" width="600">
<img src="./NB_Assets/04.05.25/_depreciated.7D.G17B.both.Coherence.png" alt="A" width="600">

## to this: 
<img src="./NB_Assets/04.05.25/_4.05.Z6.11.both.Coherence.png" alt="A" width="600">
<img src="./NB_Assets/04.05.25/_4.05.7D.G17B.both.Coherence.png" alt="A" width="600">

## For Z6.11, this greatly improve the coherences to what we'd expect. It's not perfect but it's better. However, you can see that this had no change at all for 7D.G17B.

### Exploring what is happening a bit further, let's not just look at coherences but the actual residual change in amplitudes.

### Below is a crude plot of the Power (dB) residual between corrected and original. The line in black is the station's f-notch. The line in red (covered somewhat by the notch) denotes the exact point at which this residual reaches zero. As you see, the two are in the same spot.

### You will also notice the residual (Corrected - Original) is largely positive. I'm trying to not overthink that too much as the parity shown here is highly dependent on a list of things, how the fourier transform is made (fft_win, etc) being probably the largest factor.
___
## Which leads me to my final theory to test: 
## -If the noise is fine and this coherence issue is still (to a lesser extent) happening, perhaps it is how coherence is being measured.

<img src="./NB_Assets/04.05.25/Picture1.png" alt="A" width="1200">


___
## This part of my dive into the data got very deep so I will summarize with just a few bullets of what I found and the results after I fixed the issue.
## Essentially:
- The FFT kept switching to a 5000 sample (hanning) window with a 1500 sample overlap with adjacent windows. This is likely far too large for a 2-hr trace sampled at 10hz if we are looking at two different treatments to the spectra that change at a discrete point, one region where amplitudes have changed and another region where they remain the same. It was never supposed to do this and came from a bug.
- The FFT window is now much shorter at 1750 samples with 600 sample overlap. 
    - This window was chosen using the n-log2 (aka nextpow2) best practices relation (int(np.ceil(np.log2(n=72000)))*1024).
____


# The resulting change to the coherences following this adjustment is shown below and behaves perfectly for these two stations and all others in the dataset.



<img src="./NB_Assets/04.05.25/PostCohTweak_Z6.11.both.Coherence.png" alt="A" width="600">

<img src="./NB_Assets/04.05.25/PostCohTweak_7D.G17B.both.Coherence.png" alt="A" width="600">

## Tada! Mystery solved!

## Processing all coherences using this fft-window produces the following coherences, summarized with stem-plots:
____

____
## Z6 - PLATE
<img src="./NB_Assets/04.05.25/stem_atacr/notched_PLATE.Z6.atacr.png" alt="A" width="1500">

____
## 7D - Cascadia
<img src="./NB_Assets/04.05.25/stem_atacr/notched_Cascadia.7D.atacr.png" alt="A" width="1500">

____
## 2D - Albacore
<img src="./NB_Assets/04.05.25/stem_atacr/notched_Albacore.2D.atacr.png" alt="A" width="1500">

____
## YO - ENAM
<img src="./NB_Assets/04.05.25/stem_atacr/notched_ENAM.YO.atacr.png" alt="A" width="1500">

____
## 7A - KECK
<img src="./NB_Assets/04.05.25/stem_atacr/notched_KECK.7A.atacr.png" alt="A" width="1500">

____
## YL - Lau
<img src="./NB_Assets/04.05.25/stem_atacr/notched_LAU.YL.atacr.png" alt="A" width="1500">

____
## XF - Mariana
<img src="./NB_Assets/04.05.25/stem_atacr/notched_Mariana.XF.atacr.png" alt="A" width="1500">

____
## ZA - No Melt
<img src="./NB_Assets/04.05.25/stem_atacr/notched_No_Melt.ZA.atacr.png" alt="A" width="1500">

____
## ZN - Papua
<img src="./NB_Assets/04.05.25/stem_atacr/notched_Papua.ZN.atacr.png" alt="A" width="1500">

____
## X9 - Blanco
<img src="./NB_Assets/04.05.25/stem_atacr/notched_Blanco.X9.atacr.png" alt="A" width="1500">

# Last,
___

## 2. A comprehensive source of details on the reasons for every day, station, and source-receiver pair that has been killed thus far.

## There is now 26 seperate PDFs, categorized, labeled, and with notes at the following link that detail stations issues resulting in revmoving the station or addressing another issue with it's collected noise or events.

## Additionally, the noise report shown above for the two stations examined here is repeated for all 106 stations in a single pdf located here as well.


## [LINK](https://www.zotero.org/groups/5749286/atacr.and.noisecut.comparison/collections/98CY5Z75) :[https://www.zotero.org/groups/5749286/atacr.and.noisecut.comparison/collections/98CY5Z75](https://www.zotero.org/groups/5749286/atacr.and.noisecut.comparison/collections/98CY5Z75)

